# Step 11a -- CARE-Sim Non-Causal World Model

Trains the broad non-causal CARE-Sim branch on `rl_dataset_noncausal.parquet`.

Design:
- broad dynamic state space
- broad binary action space
- static categorical embeddings
- next dynamic state + terminal + readmission heads
- no causal structural mask


In [ ]:
# Cell 1: Mount Drive + check GPU
from google.colab import drive
drive.mount("/content/drive")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: Unzip source files + verify non-causal data
import os, zipfile

DRIVE_ROOT = "/content/drive/MyDrive/CareAI"
ZIP_PATH = os.path.join(DRIVE_ROOT, "caresim_colab.zip")
UNZIP_DIR = "/content/caresim_src"
SRC_PATH = os.path.join(UNZIP_DIR, "src")
DATA_CANDIDATES = [
    os.path.join(DRIVE_ROOT, "data", "rl_dataset_noncausal.parquet"),
    os.path.join(DRIVE_ROOT, "data", "step_10_noncausal", "rl_dataset_noncausal.parquet"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)
MODEL_DIR = os.path.join(DRIVE_ROOT, "models", "icu_readmit", "caresim_noncausal")
TRAIN_SCRIPT = os.path.join(UNZIP_DIR, "scripts", "icu_readmit", "step_11a_caresim_train_noncausal.py")

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"Missing zip: {ZIP_PATH}")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(UNZIP_DIR)
for pkg_dir in [
    os.path.join(SRC_PATH, "careai"),
    os.path.join(SRC_PATH, "careai", "icu_readmit"),
    os.path.join(SRC_PATH, "careai", "icu_readmit", "caresim"),
]:
    init = os.path.join(pkg_dir, "__init__.py")
    if not os.path.exists(init):
        open(init, "w").close()
os.makedirs(MODEL_DIR, exist_ok=True)
if DATA_PATH is None:
    raise FileNotFoundError("Missing rl_dataset_noncausal.parquet under MyDrive/CareAI/data/")
print(DATA_PATH)
print(MODEL_DIR)


In [ ]:
# Cell 3: Training config
BATCH_SIZE = 32
N_EPOCHS = 20
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4
MAX_SEQ_LEN = 10
LR = 1e-3
print({
    "batch_size": BATCH_SIZE,
    "n_epochs": N_EPOCHS,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "max_seq_len": MAX_SEQ_LEN,
    "lr": LR,
})


In [ ]:
# Cell 4: Smoke test on real non-causal replay data
import subprocess, sys

def run_streaming(cmd, env):
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

SMOKE_SCRIPT = os.path.join(UNZIP_DIR, "scripts", "icu_readmit", "step_11a_caresim_noncausal_smoke_test.py")
smoke_cmd = [sys.executable, "-u", SMOKE_SCRIPT, "--data", DATA_PATH]
print(" ".join(smoke_cmd))
rc = run_streaming(smoke_cmd, env={**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'})
if rc != 0:
    raise RuntimeError('Non-causal smoke test FAILED -- do not proceed to full training.')
print('Non-causal smoke test PASSED.')


In [ ]:
# Cell 5: Train
import subprocess, sys
cmd = [
    sys.executable,
    "-u",
    TRAIN_SCRIPT,
    "--data", DATA_PATH,
    "--save-dir", MODEL_DIR,
    "--device", device,
    "--batch-size", str(BATCH_SIZE),
    "--n-epochs", str(N_EPOCHS),
    "--d-model", str(D_MODEL),
    "--n-heads", str(N_HEADS),
    "--n-layers", str(N_LAYERS),
    "--max-seq-len", str(MAX_SEQ_LEN),
    "--lr", str(LR),
]
print(" ".join(cmd))
ENV = {**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'}
rc = run_streaming(cmd, env=ENV)
if rc != 0:
    raise RuntimeError('Non-causal full training FAILED.')
print('\nNon-causal full training complete.')


In [ ]:
# Cell 6: Inspect saved artifacts
import os, json
print(sorted(os.listdir(MODEL_DIR)))
with open(os.path.join(MODEL_DIR, "train_config.json")) as f:
    cfg = json.load(f)
print(cfg["dataset"])
